# Master Pipeline: End-to-End Prediction System

## Objective
This notebook serves as the **Real-World Testing Ground**. It integrates all 4 models developed in Phases 2-5 into a single high-performance pipeline.

### Decision Pipeline:
1. **Semantic Encoder (SBERT):** Understands the ticket's intent.
2. **Category Brain:** Classifies into one of the 5 core buckets.
3. **Team/Priority Brain:** Handles routing and urgency.
4. **TTR Brain:** Estimates resolution time using a Hybrid Random Forest.
5. **Action Generation:** Retrieves the best expert instruction.

In [1]:
import pandas as pd
import numpy as np

import pickle

import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', None)
print("✅ Pipeline Environment Ready")

c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Pipeline Environment Ready


## 1. Sync All Pipeline Assets
Loading the professional model files.

In [3]:
MODELS_BASE = Path('../models')

print("🚀 Initializing Unified SBERT (384-D Vector Space)...")
sbert = SentenceTransformer('all-MiniLM-L6-v2')

print("🏷️  Syncing Category Module...")
cat_model = pickle.load(open(MODELS_BASE/'category_classifier'/'sbert_classifier.pkl', 'rb'))
cat_le = pickle.load(open(MODELS_BASE/'category_classifier'/'label_encoder.pkl', 'rb'))

print("👥 Syncing Team & Priority Modules...")
team_model = pickle.load(open(MODELS_BASE/'team_priority_classifier'/'team_classifier.pkl', 'rb'))
team_le = pickle.load(open(MODELS_BASE/'team_priority_classifier'/'le_team.pkl', 'rb'))
priority_model = pickle.load(open(MODELS_BASE/'team_priority_classifier'/'priority_classifier.pkl', 'rb'))
priority_le = pickle.load(open(MODELS_BASE/'team_priority_classifier'/'le_priority.pkl', 'rb'))

print("⏳ Syncing Resolution Time (Hybrid RF) Assets...")
ttr_model = pickle.load(open(MODELS_BASE/'resolution_time_predictor'/'ttr_model.pkl', 'rb'))
ttr_le = pickle.load(open(MODELS_BASE/'resolution_time_predictor'/'ttr_le.pkl', 'rb'))
ttr_ohe = pickle.load(open(MODELS_BASE/'resolution_time_predictor'/'ttr_ohe.pkl', 'rb'))

print("🛠️  Syncing Action Retrieval KB...")
kb_data = pickle.load(open(MODELS_BASE/'action_generator'/'action_kb_index.pkl', 'rb'))
kb_df = kb_data['knowledge_base']
kb_embeddings = kb_data['embeddings']

print("\n✨ PIPELINE SYNCHRONIZATION COMPLETE")

🚀 Initializing Unified SBERT (384-D Vector Space)...
🏷️  Syncing Category Module...
👥 Syncing Team & Priority Modules...
⏳ Syncing Resolution Time (Hybrid RF) Assets...
🛠️  Syncing Action Retrieval KB...

✨ PIPELINE SYNCHRONIZATION COMPLETE


## 2. The Multi-Objective Inference Engine

In [2]:
def run_full_pipeline(ticket_text):
    # 1. Semantic Embedding
    emb = sbert.encode([ticket_text])
    
    # 2. Get Category & Confidence
    cat_probs = cat_model.predict_proba(emb)[0]
    cat_idx = np.argmax(cat_probs)
    category = cat_le.inverse_transform([cat_idx])[0]
    confidence = cat_probs[cat_idx]
    
    # 3. Get Team & Priority
    team = team_le.inverse_transform(team_model.predict(emb))[0]
    priority = priority_le.inverse_transform(priority_model.predict(emb))[0]
    
    # 4. Get Time Bucket (Hybrid Inference)
    meta_feat = ttr_ohe.transform([[priority, category]])
    hybrid_feat = np.hstack([emb, meta_feat])
    eta = ttr_le.inverse_transform(ttr_model.predict(hybrid_feat))[0]
    
    # 5. Retrieve Smart Action
    # Uses <GENERATE_ACTION> semantic boosting
    similarities = cosine_similarity(emb, kb_embeddings).flatten()
    cat_mask = (kb_df['category'] == category).values
    similarities[cat_mask] *= 1.2 # Semantic focus
    action = kb_df.iloc[np.argmax(similarities)]['action']
    
    # 6. Routing Decision Logic
    routing = "AUTO-DISPATCH"
    if confidence < 0.75: routing = "HUMAN-REVIEW REQUIRED"
    if confidence < 0.50 or priority == 'high': routing = "URGENT ESCALATION (L2/L3)"
    
    return {
        "Category": f"{category} ({confidence:.1%})",
        "Assigned Team": team,
        "Priority": priority.upper(),
        "ETA Range": eta,
        "Routing Status": routing,
        "Generated Action": action
    }

print("✅ Inference Engine Loaded")

✅ Inference Engine Loaded


## 3. REAL-WORLD TESTING GROUND (The Dashboard View)
Run this cell to test **ANY** input text.

In [4]:
test_input = "My internet speed is very slow and I need a refund for the downtime."

print(f"🎫 TESTING INPUT: {test_input}\n")
results = run_full_pipeline(test_input)

for key, val in results.items():
    print(f"{key:20}: {val}")

🎫 TESTING INPUT: My internet speed is very slow and I need a refund for the downtime.

Category            : Billing & Refunds (59.3%)
Assigned Team       : Financial Operations
Priority            : HIGH
ETA Range           : < 1 Hour
Routing Status      : URGENT ESCALATION (L2/L3)
Generated Action    : Sorry for the inconvenience, could you please share details and your current status for better assistance?


c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


## 4. Batch Stress Test
Testing the edge cases from the prompt.

In [5]:
examples = [
    "I want to cancel my premium plan and get my 99 dollars back.",
    "The application is crashing on my iPhone 15 when I click upload.",
    "Double charge alert: My credit card was charged twice today.",
    "How do I update the shipping address for order #12345?"
]

final_results = []
for ex in examples:
    final_results.append({"Input": ex, **run_full_pipeline(ex)})

pd.DataFrame(final_results)

c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\SRINATH\Desktop\data science\machine learing\ml project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Input,Category,Assigned Team,Priority,ETA Range,Routing Status,Generated Action
0,I want to cancel my premium plan and get my 99 dollars back.,Billing & Refunds (92.5%),Financial Operations,HIGH,< 1 Hour,URGENT ESCALATION (L2/L3),"Please check account <acc_num> to update billing information, which should help resolve the issue. For further assistance, please call <tel_num> to complete the process."
1,The application is crashing on my iPhone 15 when I click upload.,Technical Support (52.3%),Technical Engineering,HIGH,< 1 Hour,URGENT ESCALATION (L2/L3),"For a more detailed analysis, please provide additional information or contact us for further assistance."
2,Double charge alert: My credit card was charged twice today.,Billing & Refunds (61.0%),Financial Operations,HIGH,< 1 Hour,URGENT ESCALATION (L2/L3),"Dear [Name], we apologize for the billing issues you are experiencing with double charges on recent subscriptions. We would like to assist in resolving this matter. To better understand the issue, could you please provide details on your account settings and any interactions with support? We prefer to schedule a call at [Tel Number]. Please let us know a convenient time for discussing your account [Acc Num]."
3,How do I update the shipping address for order #12345?,Billing & Refunds (75.2%),Financial Operations,LOW,> 24 Hours,AUTO-DISPATCH,"We are happy to assist with updating the billing details for the digital strategy tools and services. To complete this request, we need to verify the information. Please provide your company's account number associated with the services and any new billing address or payment method details. We prefer to receive this information via email, but if that is not convenient, we can discuss further at a time that is convenient for you. Please let us know an option that suits you so we can update the billing details promptly and avoid any delays."
